Cell 1 — Mount Google Drive

In [ ]:
# ============================================================
# CELL 1: Mount Google Drive
# Experiment X2 - XLS-R 300M corrected fine-tuning
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


Cell 2 — Define project paths

In [ ]:
# ============================================================
# CELL 2: Define Corpus V1.1 and X2 paths
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm"
)

TRAIN_CSV = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "corpus_v1_1"
    / "train.csv"
)

VALIDATION_CSV = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "corpus_v1_1"
    / "validation.csv"
)

VOCAB_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "xlsr_tokenizer_v1_1"
)

VOCAB_PATH = VOCAB_DIR / "vocab.json"

XLSR_DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "xlsr_corpus_v1_1"
)

X2_OUTPUT_DIR = (
    PROJECT_ROOT
    / "models"
    / "xlsr_300m_X2_corpus_v1_1"
)

print("Project:", PROJECT_ROOT.exists())
print("Train V1.1:", TRAIN_CSV.exists())
print("Validation V1.1:", VALIDATION_CSV.exists())

Project: True
Train V1.1: True
Validation V1.1: True


In [ ]:
# ============================================================
# CELL 4: Load Corpus V1.1 train and validation sets
# ============================================================

import pandas as pd
from datasets import Dataset, DatasetDict

train_df = pd.read_csv(TRAIN_CSV)
validation_df = pd.read_csv(VALIDATION_CSV)

dataset = DatasetDict({
    "train": Dataset.from_pandas(
        train_df,
        preserve_index=False
    ),
    "validation": Dataset.from_pandas(
        validation_df,
        preserve_index=False
    ),
})

print(dataset)

print("\nTrain duration:",
      round(train_df["duration_seconds"].sum() / 3600, 3),
      "hours")

print("Validation duration:",
      round(validation_df["duration_seconds"].sum() / 60, 2),
      "minutes")

DatasetDict({
    train: Dataset({
        features: ['segment_id', 'recording_id', 'speaker_group_id', 'audio_path', 'duration_seconds', 'transcription', 'absolute_audio_path'],
        num_rows: 1472
    })
    validation: Dataset({
        features: ['segment_id', 'recording_id', 'speaker_group_id', 'audio_path', 'duration_seconds', 'transcription', 'absolute_audio_path'],
        num_rows: 133
    })
})

Train duration: 4.638 hours
Validation duration: 18.12 minutes


In [ ]:
# ============================================================
# CELL 4: Verify Corpus V1.1 character inventory
# ============================================================

all_text = (
    train_df["transcription"].astype(str).tolist()
    + validation_df["transcription"].astype(str).tolist()
)

characters = sorted(
    set("".join(all_text))
)

print("Characters:")
print(characters)

print("\nř present:", "ř" in characters)
print("r present:", "r" in characters)

Characters:
[' ', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'p', 'q', 'r', 's', 't', 'u', 'w', 'x', 'y', 'z', 'ǧ', 'ɛ', 'ɣ', 'ʷ', 'ḍ', 'ḥ', 'ṣ', 'ṭ', 'ẓ']

ř present: False
r present: True


In [ ]:
# ============================================================
# CELL 5: Build Corpus V1.1 Tarifit CTC vocabulary
# ============================================================

import json

vocab_dict = {
    char: idx
    for idx, char in enumerate(characters)
}

# Space -> word delimiter
space_id = vocab_dict.pop(" ")
vocab_dict["|"] = space_id

# Special tokens
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

print("Vocabulary size:", len(vocab_dict))
print(vocab_dict)

Vocabulary size: 36
{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'p': 15, 'q': 16, 'r': 17, 's': 18, 't': 19, 'u': 20, 'w': 21, 'x': 22, 'y': 23, 'z': 24, 'ǧ': 25, 'ɛ': 26, 'ɣ': 27, 'ʷ': 28, 'ḍ': 29, 'ḥ': 30, 'ṣ': 31, 'ṭ': 32, 'ẓ': 33, '|': 0, '[UNK]': 34, '[PAD]': 35}


In [ ]:
# ============================================================
# CELL 6: Save Corpus V1.1 XLS-R vocabulary
# ============================================================

VOCAB_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "xlsr_tokenizer_v1_1"
)

VOCAB_DIR.mkdir(
    parents=True,
    exist_ok=True
)

VOCAB_PATH = VOCAB_DIR / "vocab.json"

with open(
    VOCAB_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        vocab_dict,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved:")
print(VOCAB_PATH)

Saved:
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/xlsr_tokenizer_v1_1/vocab.json


In [ ]:
# ============================================================
# CELL 7: Create Corpus V1.1 tokenizer and XLS-R processor
# ============================================================

from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
)

tokenizer = Wav2Vec2CTCTokenizer(
    str(VOCAB_PATH),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)

print("Tokenizer size:", len(tokenizer))
print("ř in vocab:", "ř" in tokenizer.get_vocab())

Tokenizer size: 38
ř in vocab: False


Cell 8 — Test the new tokenizer

In [ ]:
# ============================================================
# CELL 8: Test Corpus V1.1 tokenizer
# ============================================================

sample_text = train_df.iloc[0]["transcription"]

encoded = tokenizer(sample_text)

decoded = tokenizer.decode(
    encoded.input_ids,
    group_tokens=False
)

print("Original:")
print(sample_text)

print("\nToken IDs:")
print(encoded.input_ids)

print("\nDecoded:")
print(decoded)

print("\nExact match:", sample_text == decoded)

Original:
rexbar asbḥan n yasuɛ lmasiḥ sufuss n matta

Token IDs:
[17, 5, 22, 2, 1, 17, 0, 1, 18, 2, 30, 1, 14, 0, 14, 0, 23, 1, 18, 20, 26, 0, 12, 13, 1, 18, 9, 30, 0, 18, 20, 6, 20, 18, 18, 0, 14, 0, 13, 1, 19, 19, 1]

Decoded:
rexbar asbḥan n yasuɛ lmasiḥ sufuss n matta

Exact match: True


Cell 9 — Install dependencies

In [ ]:
# ============================================================
# CELL 9: Install dependencies
# ============================================================

!pip install -q transformers datasets accelerate jiwer soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 110.8 MB/s eta 0:00:00


Cell 10 — Define XLS-R V1.1 preprocessing

In [ ]:
# ============================================================
# CELL 10: Define XLS-R preprocessing for Corpus V1.1
# ============================================================

import soundfile as sf


def prepare_xlsr_dataset(example):

    audio_file = (
        PROJECT_ROOT
        / example["audio_path"]
    )

    audio, sampling_rate = sf.read(
        audio_file
    )

    # Audio waveform -> normalized XLS-R input
    inputs = processor(
        audio,
        sampling_rate=sampling_rate
    )

    example["input_values"] = (
        inputs.input_values[0]
    )

    example["input_length"] = len(
        example["input_values"]
    )

    # V1.1 transcription -> CTC character IDs
    example["labels"] = tokenizer(
        example["transcription"]
    ).input_ids

    return example


print("Preprocessing function ready.")

Preprocessing function ready.


Cell 11 — Test preprocessing on one example

In [ ]:
# ============================================================
# CELL 11: Test XLS-R V1.1 preprocessing on one segment
# ============================================================

sample = prepare_xlsr_dataset(
    dataset["train"][0]
)

print("Input samples:", len(sample["input_values"]))
print("Input length:", sample["input_length"])

print("\nReference:")
print(sample["transcription"])

print("\nDecoded labels:")
print(
    tokenizer.decode(
        sample["labels"],
        group_tokens=False
    )
)

Input samples: 60160
Input length: 60160

Reference:
rexbar asbḥan n yasuɛ lmasiḥ sufuss n matta

Decoded labels:
rexbar asbḥan n yasuɛ lmasiḥ sufuss n matta


Cell 12: Preprocess full XLS-R Corpus V1.1

In [ ]:
# ============================================================
# CELL 12: Preprocess full XLS-R Corpus V1.1
# ============================================================

xlsr_dataset = dataset.map(
    prepare_xlsr_dataset,
    remove_columns=dataset["train"].column_names,
    num_proc=1
)

print(xlsr_dataset)

Map:   0%|          | 0/1472 [00:00<?, ? examples/s]

Map:   0%|          | 0/133 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_values', 'input_length', 'labels'],
        num_rows: 1472
    })
    validation: Dataset({
        features: ['input_values', 'input_length', 'labels'],
        num_rows: 133
    })
})


Cell 13 — Save the processed V1.1 dataset

In [ ]:
# ============================================================
# CELL 13: Save preprocessed XLS-R Corpus V1.1
# ============================================================

xlsr_dataset.save_to_disk(
    str(XLSR_DATASET_PATH)
)

print("Saved to:")
print(XLSR_DATASET_PATH)

Saving the dataset (0/3 shards):   0%|          | 0/1472 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/133 [00:00<?, ? examples/s]

Saved to:
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/xlsr_corpus_v1_1


Cell 14 — Define the CTC data collator

In [ ]:
# ============================================================
# CELL 14: Define XLS-R CTC data collator
# ============================================================

from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch


@dataclass
class DataCollatorCTCWithPadding:
    processor: Any
    padding: Union[bool, str] = True

    def __call__(
        self,
        features: List[
            Dict[
                str,
                Union[List[int], torch.Tensor]
            ]
        ]
    ) -> Dict[str, torch.Tensor]:

        # Audio inputs
        input_features = [
            {
                "input_values":
                feature["input_values"]
            }
            for feature in features
        ]

        # CTC label sequences
        label_features = [
            {
                "input_ids":
                feature["labels"]
            }
            for feature in features
        ]

        # Dynamically pad audio
        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        # Dynamically pad text labels
        labels_batch = (
            self.processor.tokenizer.pad(
                label_features,
                padding=self.padding,
                return_tensors="pt",
            )
        )

        # Ignore padding in CTC loss
        labels = (
            labels_batch["input_ids"]
            .masked_fill(
                labels_batch.attention_mask.ne(1),
                -100
            )
        )

        batch["labels"] = labels

        return batch


data_collator = DataCollatorCTCWithPadding(
    processor=processor,
    padding=True,
)

print("CTC data collator ready.")

CTC data collator ready.


Cell 14 — Define the CTC data collator

In [ ]:
# ============================================================
# CELL 14: Define XLS-R CTC data collator
# ============================================================

from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch


@dataclass
class DataCollatorCTCWithPadding:
    processor: Any
    padding: Union[bool, str] = True

    def __call__(
        self,
        features: List[
            Dict[
                str,
                Union[List[int], torch.Tensor]
            ]
        ]
    ) -> Dict[str, torch.Tensor]:

        # Audio inputs
        input_features = [
            {
                "input_values":
                feature["input_values"]
            }
            for feature in features
        ]

        # CTC label sequences
        label_features = [
            {
                "input_ids":
                feature["labels"]
            }
            for feature in features
        ]

        # Dynamically pad audio
        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        # Dynamically pad text labels
        labels_batch = (
            self.processor.tokenizer.pad(
                label_features,
                padding=self.padding,
                return_tensors="pt",
            )
        )

        # Ignore padding in CTC loss
        labels = (
            labels_batch["input_ids"]
            .masked_fill(
                labels_batch.attention_mask.ne(1),
                -100
            )
        )

        batch["labels"] = labels

        return batch


data_collator = DataCollatorCTCWithPadding(
    processor=processor,
    padding=True,
)

print("CTC data collator ready.")

CTC data collator ready.


Cell 15 — Test one batch

In [ ]:
# ============================================================
# CELL 15: Test XLS-R V1.1 training batch
# ============================================================

test_batch = data_collator([
    xlsr_dataset["train"][0],
    xlsr_dataset["train"][1],
])

print(
    "Input values:",
    test_batch["input_values"].shape
)

print(
    "Attention mask:",
    test_batch["attention_mask"].shape
)

print(
    "Labels:",
    test_batch["labels"].shape
)

print(
    "Batch keys:",
    test_batch.keys()
)

Cell 16 — Define WER and CER

In [ ]:
# ============================================================
# CELL 16: Define XLS-R WER and CER metrics
# ============================================================

import numpy as np
from jiwer import wer, cer


def compute_metrics(pred):

    # Logits -> most likely CTC token
    pred_ids = np.argmax(
        pred.predictions,
        axis=-1
    )

    label_ids = pred.label_ids.copy()

    label_ids[label_ids == -100] = (
        processor.tokenizer.pad_token_id
    )

    # CTC prediction decoding:
    # repeated predicted symbols are collapsed
    pred_str = processor.batch_decode(
        pred_ids
    )

    # Ground truth must preserve genuine repeated letters
    label_str = processor.batch_decode(
        label_ids,
        group_tokens=False
    )

    return {
        "wer": wer(
            label_str,
            pred_str
        ) * 100,

        "cer": cer(
            label_str,
            pred_str
        ) * 100,
    }


print("WER/CER metrics ready.")

Cell 17 — Load a fresh XLS-R 300M model for X2

In [ ]:
# ============================================================
# CELL 17: Load fresh XLS-R 300M for Experiment X2
# ============================================================

from transformers import Wav2Vec2ForCTC

MODEL_NAME = (
    "facebook/wav2vec2-xls-r-300m"
)

model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,

    # New output vocabulary
    vocab_size=len(tokenizer),

    # CTC configuration
    pad_token_id=tokenizer.pad_token_id,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,

    # X2 regularization setup
    attention_dropout=0.0,
    hidden_dropout=0.0,
    feat_proj_dropout=0.0,
    mask_time_prob=0.05,
    layerdrop=0.0,
)

print("Model:", MODEL_NAME)
print(
    "Vocabulary size:",
    model.config.vocab_size
)

print(
    "Parameters:",
    round(
        model.num_parameters() / 1e6,
        1
    ),
    "M"
)

config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.27GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
project_hid.weight           | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
lm_head.bias                 | MISSING    | 
lm_head.weight               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model: facebook/wav2vec2-xls-r-300m
Vocabulary size: 38
Parameters: 315.5 M


Cell 18 — Freeze the convolutional feature encoder

In [ ]:
# ============================================================
# CELL 18: Freeze XLS-R convolutional feature encoder
# ============================================================

model.freeze_feature_encoder()

print("Feature encoder frozen.")

Cell 19 — Check GPU

In [ ]:
# ============================================================
# CELL 19: Check GPU before X2 training
# ============================================================

import torch

print(
    "CUDA available:",
    torch.cuda.is_available()
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    free_memory, total_memory = (
        torch.cuda.mem_get_info()
    )

    print(
        "Total GPU memory:",
        round(
            total_memory / 1024**3,
            2
        ),
        "GB"
    )

    print(
        "Free GPU memory:",
        round(
            free_memory / 1024**3,
            2
        ),
        "GB"
    )

Cell 20 — Configure X2 training

In [ ]:
# ============================================================
# CELL 20: Configure XLS-R Experiment X2
# ============================================================

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=str(X2_OUTPUT_DIR),

    # --------------------------------------------------------
    # Batch configuration for Tesla T4
    # --------------------------------------------------------
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,

    per_device_eval_batch_size=2,

    # Effective batch size = 2 x 4 = 8

    # --------------------------------------------------------
    # Group similar audio lengths together
    # --------------------------------------------------------
    group_by_length=True,
    length_column_name="input_length",

    # --------------------------------------------------------
    # X2 optimization
    # --------------------------------------------------------
    learning_rate=1e-4,
    weight_decay=0.005,
    warmup_steps=100,

    num_train_epochs=10,

    # --------------------------------------------------------
    # GPU / memory
    # --------------------------------------------------------
    fp16=True,
    gradient_checkpointing=True,

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------
    eval_strategy="epoch",

    # --------------------------------------------------------
    # Checkpoints
    # --------------------------------------------------------
    save_strategy="epoch",
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    # --------------------------------------------------------
    # Logging
    # --------------------------------------------------------
    logging_strategy="steps",
    logging_steps=25,

    report_to="none",

    # --------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------
    seed=42,
)

print("X2 training arguments ready.")

Cell 21 — Create the X2 Trainer

In [ ]:
# ============================================================
# CELL 21: Create XLS-R X2 Trainer
# ============================================================

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=(
        xlsr_dataset["train"]
    ),

    eval_dataset=(
        xlsr_dataset["validation"]
    ),

    data_collator=data_collator,

    compute_metrics=compute_metrics,
)

print("X2 Trainer ready.")

Cell 22 — Final sanity check

In [ ]:
# ============================================================
# CELL 22: Final X2 sanity check before training
# ============================================================

print(
    "Train examples:",
    len(xlsr_dataset["train"])
)

print(
    "Validation examples:",
    len(xlsr_dataset["validation"])
)

print(
    "Tokenizer size:",
    len(tokenizer)
)

print(
    "Model vocabulary size:",
    model.config.vocab_size
)

print(
    "ř in tokenizer:",
    "ř" in tokenizer.get_vocab()
)

print(
    "Learning rate:",
    training_args.learning_rate
)

print(
    "Epochs:",
    training_args.num_train_epochs
)

print(
    "CUDA:",
    torch.cuda.is_available()
)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

Cell 23 — Start XLS-R X2 fine-tuning

In [ ]:
# ============================================================
# CELL 23: Start XLS-R 300M Experiment X2
# ============================================================

train_result = trainer.train()

Cell 24 — Display the best X2 checkpoint

In [ ]:
# ============================================================
# CELL 24: Show best X2 checkpoint and WER
# ============================================================

print(
    "Best checkpoint:",
    trainer.state.best_model_checkpoint
)

print(
    "Best validation WER:",
    trainer.state.best_metric
)

Cell 25 — Save final processor with X2

In [ ]:
# ============================================================
# CELL 25: Save V1.1 processor with X2 model
# ============================================================

FINAL_PROCESSOR_DIR = (
    X2_OUTPUT_DIR
    / "processor"
)

processor.save_pretrained(
    str(FINAL_PROCESSOR_DIR)
)

print("Processor saved to:")
print(FINAL_PROCESSOR_DIR)